In [1]:
from pathlib import Path
import pandas as pd
import yfinance as yf

In [2]:
port_tickers = ['AAPL','MSFT','NVDA','GOOGL','AMZN','TSLA','JPM','JNJ','UNH','XOM','PG','CAT','NEE']
bench = 'SPY'
tickers = sorted(port_tickers + [bench])

In [3]:
tickers

['AAPL',
 'AMZN',
 'CAT',
 'GOOGL',
 'JNJ',
 'JPM',
 'MSFT',
 'NEE',
 'NVDA',
 'PG',
 'SPY',
 'TSLA',
 'UNH',
 'XOM']

In [4]:
start_date = '2018-01-01'
end_date   = '2024-12-31'

raw_dir = Path('data/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

out_prices = raw_dir / 'prices_adjclose.parquet'
out_meta   = raw_dir / 'meta_tickers.csv'
out_regime = raw_dir / 'market_regimes.csv'

# === tải dữ liệu Adj Close ===
print('Downloading Close:', tickers)
df = yf.download(
    tickers=tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False,
    group_by='column'
)

In [5]:
# lấy đúng lớp "Adj Close" (MultiIndex) hoặc cột đơn nếu chỉ 1 ticker
if isinstance(df.columns, pd.MultiIndex):
    df = df['Adj Close'].copy()
else:
    # trường hợp hiếm: một ticker -> DataFrame 1 cấp
    df = df[['Adj Close']].rename(columns={'Adj Close': tickers[0]})

In [6]:
# chuẩn hóa index ngày
df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
df = df[~df.index.duplicated(keep='last')]
df = df.reindex(sorted(df.columns), axis=1)

In [7]:
# báo cáo coverage nhanh
n = len(df)
rep = pd.DataFrame({
    'missing': df.isna().sum(),
    'coverage_pct': (1 - df.isna().sum()/n)*100
}).round(2)
print('\n[Coverage report]')
print(rep)

# lưu parquet
df.to_parquet(out_prices)
print(f'\n✓ saved raw prices -> {out_prices.resolve()}')

# tạo file meta tickers ↔ sector (bạn điền tay sau)
if not out_meta.exists():
    pd.DataFrame({'ticker': port_tickers, 'sector': ''}).to_csv(out_meta, index=False)
    print(f'✓ created meta template -> {out_meta.resolve()}')

# tạo file phân kỳ thị trường (có thể chỉnh sửa sau)
if not out_regime.exists():
    regimes = pd.DataFrame([
        ['2018-01-01','2020-02-19','Pre-COVID Bull'],
        ['2020-02-20','2020-12-31','COVID Shock/Recovery'],
        ['2021-01-01','2022-12-31','Inflation & Tightening'],
        ['2023-01-01','2024-12-31','AI-led Recovery'],
    ], columns=['start','end','regime'])
    regimes.to_csv(out_regime, index=False)
    print(f'✓ created regime file -> {out_regime.resolve()}')

print('\nNext: run 02_make_processed.py')


[Coverage report]
        missing  coverage_pct
Ticker                       
AAPL          0         100.0
AMZN          0         100.0
CAT           0         100.0
GOOGL         0         100.0
JNJ           0         100.0
JPM           0         100.0
MSFT          0         100.0
NEE           0         100.0
NVDA          0         100.0
PG            0         100.0
SPY           0         100.0
TSLA          0         100.0
UNH           0         100.0
XOM           0         100.0

✓ saved raw prices -> C:\Users\ACER\python\Dissertation\Port Dashboard\data\raw\prices_adjclose.parquet

Next: run 02_make_processed.py


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

p = Path("data/raw/prices_adjclose.parquet")
df = pd.read_parquet(p)

print("Shape:", df.shape)
print("Columns:", list(df.columns))
assert df.index.is_monotonic_increasing, "Index phải tăng dần"
assert df.index.is_unique, "Index phải unique"
assert df.index.tz is None, "Index nên tz-naive (None)"
assert 'SPY' in df.columns, "Thiếu SPY"
assert (df > 0).all().all(), "Có giá <= 0"

n = len(df)
miss = df.isna().sum().sum()
cov = 1 - miss / (n * df.shape[1])
print(f"Missing cells: {miss} ({(1-cov)*100:.2f}% of all)")
assert cov >= 0.98, "Coverage < 98% → check lại tải dữ liệu"

dup = df.index.duplicated().sum()
assert dup == 0, "Có ngày bị duplicate"

print("RAW: OK ✅")

Shape: (1760, 14)
Columns: ['AAPL', 'AMZN', 'CAT', 'GOOGL', 'JNJ', 'JPM', 'MSFT', 'NEE', 'NVDA', 'PG', 'SPY', 'TSLA', 'UNH', 'XOM']
Missing cells: 0 (0.00% of all)
RAW: OK ✅
